In [ ]:
import os
import pandas as pd
import numpy as np
import sys
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.append(('/Users/blancamartin/Desktop/Voytek_Lab/spike_waveform/spikeparam/AP_empirical_paper1/datasets/spe-1/spe1_helper_modules/'))
import importlib, spk_feat_cluster_comp_analysis
importlib.reload(spk_feat_cluster_comp_analysis)
from spk_feat_cluster_comp_analysis import (
    find_temporal_transitions,
    plot_temporal_transitions,
    plot_temporal_transitions_highlights,
)
import config
from config import SPE1_PICKLE_ROOT

In [ ]:
cluster_pickle_dir = "/Users/blancamartin/Desktop/Voytek_Lab/spike_waveform/spe1_pickles/cluster_pickles/"

## Temporal transition times

For cells where cluster membership has a strong temporal component (|ρ| > 0.3, p < 0.05), find the recording time at which the cluster label transitions using a sigmoid fit to the rolling mean of ordinal cluster labels.

**Method**: fit a 4-parameter logistic `y = b + L / (1 + exp(-k·(t − t₀)))` to the rolling mean cluster label time series.  The inflection point **t₀** is the transition time; **k** (1/s) captures sharpness — large |k| = abrupt switch, small |k| = gradual drift.  Falls back to time-series midpoint when the fit fails.

In [ ]:
transition_save_path = os.path.join(config.SPE1_PICKLE_ROOT, 'cluster_pickles', 'temporal_transitions.pkl')

df_transitions = find_temporal_transitions(
    cluster_pickle_dir,
    rho_thresh=0.3,
    p_thresh=0.05,
    min_frac=0.1,
    rolling_n=50,
    save_path=transition_save_path,
)
df_transitions

In [ ]:
plot_temporal_transitions(df_transitions, cluster_pickle_dir, n_cols=4, rolling_n=50)

In [ ]:
highlight_selections = [
    ('c42', 'peak_width'),
    ('c20', 'peak_amp'),
    ('c3',  'peak_amp'),
    ('c45', 'exp_lambda'),
    ('c7',  'peak_sharpness'),
    ('c7',  'inflection_time'),
]

plot_temporal_transitions_highlights(df_transitions, cluster_pickle_dir,
                                     highlight_selections, n_cols=3, rolling_n=50)

A positive correlation between temporal ρ and nRMSE is a warning: cells where cluster membership is most time-dependent also look most different between clusters — but that difference may just reflect recording instability, not biology.  This is why we explicitly select high-difference / time-independent cells for priority LFP analysis.